In [137]:
import pandas as pd
from tqdm import tqdm

In [138]:
FILE_AGRESTE_PATH = '../../../data/external_data/agreste/'
ENTREPOT_PATH = '~/Bureau/utils/data/'

df = {}

In [ ]:
df['agreste_ift_gcpe_reference_region_2017'] = pd.read_csv(FILE_AGRESTE_PATH+'final/agreste_ift_gcpe_reference_region_2017.csv')
df['agreste_ift_gcpe_reference_region_2021'] = pd.read_csv(FILE_AGRESTE_PATH+'final/agreste_ift__gcpe_reference_region_2021.csv')

In [140]:
def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)


tables_entrepot = [
    'sdc', 'domaine', 'dispositif', 'synthetise', 'commune'
]
tables_without_id = [
    'sdc_realise_performance', 'synthetise_synthetise_performance', 
    'entite_unique_par_sdc_nettoyage', 
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_entrepot, ENTREPOT_PATH, sep = ',',index_col='id',verbose=False) 
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',',verbose=False) 

100%|██████████| 3/3 [00:06<00:00,  2.11s/it]


In [141]:
left = df['sdc'].reset_index()
right = df['entite_unique_par_sdc_nettoyage']
df['sdc_extanded'] = pd.merge(left, right, left_on='id', right_on='sdc_id', how='left').set_index('id')

In [142]:
# sdc_realise_id = list(df['sdc_extanded'].loc[
#     df['sdc_extanded']['entite_retenue'] == 'realise_retenu'
# ].sample(10).index)

# sdc_synthetise_id = list(df['sdc_extanded'].loc[
#     (df['sdc_extanded']['entite_retenue'] != 'realise_retenu') & 
#     ~(df['sdc_extanded']['entite_retenue'].isna())
# ].sample(10).index)

sdc_realise_id = ['fr.inra.agrosyst.api.entities.GrowingSystem_768b1063-c8c5-46c6-a307-5561788e00e3',
 'fr.inra.agrosyst.api.entities.GrowingSystem_cdbfcbfc-db55-43b3-bce7-96f01085aa2d',
 'fr.inra.agrosyst.api.entities.GrowingSystem_89443893-b1be-4f88-ab8b-e966f64a7fc7',
 'fr.inra.agrosyst.api.entities.GrowingSystem_8b19ba79-e143-496d-9daf-1746d3538605',
 'fr.inra.agrosyst.api.entities.GrowingSystem_4d5b4d2a-9ebe-4b53-84a5-1cc40fe5856b',
 'fr.inra.agrosyst.api.entities.GrowingSystem_0e0e4756-bbf5-4c87-a287-c7b13839c8f3',
 'fr.inra.agrosyst.api.entities.GrowingSystem_5b2752f7-4d51-4cf0-97f7-f1d16f146e57',
 'fr.inra.agrosyst.api.entities.GrowingSystem_134e6133-c67e-4a57-9c78-772f11594073',
 'fr.inra.agrosyst.api.entities.GrowingSystem_d004c218-d0fb-4f48-a880-2eba6d0849cb',
 'fr.inra.agrosyst.api.entities.GrowingSystem_a3bfad12-07cb-4c95-94cb-a0bdee864e4a']

sdc_synthetise_id = ['fr.inra.agrosyst.api.entities.GrowingSystem_48729789-f3d8-42aa-b384-784b0859de84',
 'fr.inra.agrosyst.api.entities.GrowingSystem_4c799848-ad94-4f9f-82fa-ef919730bb3d',
 'fr.inra.agrosyst.api.entities.GrowingSystem_ca60b736-6189-4e22-92d0-6e46f8997d08',
 'fr.inra.agrosyst.api.entities.GrowingSystem_6321f4b9-fced-4ef3-9d38-a5c9be032013',
 'fr.inra.agrosyst.api.entities.GrowingSystem_7bcd9392-07d6-4f2e-94b2-41935b0a0dc6',
 'fr.inra.agrosyst.api.entities.GrowingSystem_0b39e0f4-3846-462e-9fd5-2eb4f587d7b4',
 'fr.inra.agrosyst.api.entities.GrowingSystem_15a85b1b-f708-4295-a974-cbbf7e2a6362',
 'fr.inra.agrosyst.api.entities.GrowingSystem_29a4a164-9f10-44b4-9623-15878bc4dfba',
 'fr.inra.agrosyst.api.entities.GrowingSystem_ab77ea61-b574-4dde-8cd8-9040f3cd7f80',
 'fr.inra.agrosyst.api.entities.GrowingSystem_2a3f7516-705e-4d59-8bcb-1c5a75515941']

STUDIED_IDS = sdc_realise_id + sdc_synthetise_id

In [143]:
df['sdc_test'] = df['sdc'].loc[
    STUDIED_IDS
]

df['sdc_realise_performance_test'] = df['sdc_realise_performance'].loc[
    df['sdc_realise_performance']['sdc_id'].isin(STUDIED_IDS)
]
df['synthetise_test'] = df['synthetise'].loc[
    df['synthetise']['sdc_id'].isin(
        STUDIED_IDS
    )
]
df['synthetise_synthetise_performance_test'] = df['synthetise_synthetise_performance'].loc[
    df['synthetise_synthetise_performance']['synthetise_id'].isin(df['synthetise_test'].index)
]
df['dispositif_test'] = df['dispositif'].loc[
    df['dispositif'].index.isin(df['sdc_test']['dispositif_id'])
]

df['domaine_test'] = df['domaine'].loc[
    df['domaine'].index.isin(df['dispositif_test']['domaine_id'])
]
df['entite_unique_par_sdc_nettoyage_test'] = df['entite_unique_par_sdc_nettoyage'].loc[
    df['entite_unique_par_sdc_nettoyage']['sdc_id'].isin(STUDIED_IDS)
]

df['commune_test'] = df['commune'].loc[
    df['commune'].index.isin(df['domaine_test']['commune_id'])
]

## anonymisation

In [144]:
df['domaine_test'].loc[:, 'nom'] = 'anonyme'
df['domaine_test'].loc[:, 'siret'] = 'anonyme'
df['domaine_test'].loc[:, 'contact_principal'] = 'anonyme'
df['domaine_test'].loc[:, 'annee_naissance_exploitant'] = 'anonyme'
df['domaine_test'].loc[:, 'responsables_domaine'] = 'anonyme'

df['dispositif_test'].loc[:, 'nom'] = 'anonyme'

df['sdc_test'].loc[:, 'reseaux_ir'] = 'anonyme'
df['sdc_test'].loc[:, 'reseaux_it'] = 'anonyme'
df['sdc_test'].loc[:, 'nom'] = 'anonyme'

df['synthetise_test'].loc[:, 'nom'] = 'anonyme'


df['commune_test'] = df['commune_test'][['ancienne_region']]

/tmp/ipykernel_91832/505582540.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'anonyme' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df['domaine_test'].loc[:, 'siret'] = 'anonyme'
/tmp/ipykernel_91832/505582540.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'anonyme' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df['domaine_test'].loc[:, 'annee_naissance_exploitant'] = 'anonyme'


## Export des données

In [ ]:
path='./'
df['sdc_test'].to_csv(path+'sdc.csv')
df['sdc_realise_performance_test'].to_csv(path+'sdc_realise_performance.csv')
df['synthetise_test'].to_csv(path+'synthetise'+'.csv')
df['synthetise_synthetise_performance_test'].to_csv(path+'synthetise_synthetise_performance'+'.csv')
df['dispositif_test'].to_csv(path+'dispositif'+'.csv')
df['domaine_test'].to_csv(path+'domaine'+'.csv')
df['entite_unique_par_sdc_nettoyage_test'].to_csv(path+'entite_unique_par_sdc_nettoyage'+'.csv')
df['commune_test'].to_csv(path+'commune'+'.csv')
df['entite_unique_par_sdc_nettoyage_test'].to_csv(path+'entite_unique_par_sdc_nettoyage'+'.csv')
df['agreste_ift_gcpe_reference_region_2017'].set_index('nom_ancienne_region').to_csv(path+'agreste_ift_gcpe_reference_region_2017'+'.csv')
df['agreste_ift_gcpe_reference_region_2021'].set_index('nom_ancienne_region').to_csv(path+'agreste_ift_gcpe_reference_region_2021'+'.csv')

In [339]:
df['agreste_ift_reference_region_2017'].set_index('nom_ancienne_region')

,contribution_ift_corrigee
nom_ancienne_region,
Alsace,3.04
Aquitaine,2.68
Auvergne,3.23
Basse-Normandie,4.29
Bourgogne,4.61
Bretagne,3.62
Centre-Val de Loire,5.20
Champagne-Ardenne,5.37
Franche-Comté,4.53


## Résolution du problème

In [165]:
df['agreste_ift_reference_region_2017'][['contribution_ift_corrigee']]

,contribution_ift_corrigee
0,3.04
1,2.68
2,3.23
3,4.29
4,4.61
5,3.62
6,5.20
7,5.37
8,4.53
9,1.70


In [ ]:
#=========#
# commmun #
#=========#
# on ajoute les informations au sdc (notamment l'ancienne région)
left = df['sdc'][['campagne', 'filiere', 'dispositif_id']]
right = df['dispositif'][['domaine_id']]
df['sdc_extanded'] = pd.merge(left, right, left_on = 'dispositif_id', right_index=True, how='left')

left = df['sdc_extanded']
right = df['domaine'][['commune_id']]
df['sdc_extanded'] = pd.merge(left, right, left_on = 'domaine_id', right_index=True, how='left')

left = df['sdc_extanded']
right = df['commune'][['ancienne_region']]
df['sdc_extanded'] = pd.merge(left, right, left_on = 'commune_id', right_index=True, how='left')

left = df['sdc_extanded'].reset_index()
right = df['entite_unique_par_sdc_nettoyage']
df['sdc_extanded'] = pd.merge(left, right, left_on='id', right_on='sdc_id', how='left').set_index('id')

# on ajoute les informations issues d'Agreste
# pour 2017
left = df['sdc_extanded'][['campagne', 'ancienne_region']].loc[
    df['sdc_extanded']['campagne'] == 2017
].reset_index()
right = df['agreste_ift_gcpe_reference_region_2017'][['nom_ancienne_region', 'contribution_ift_corrigee']]
df['sdc_extanded_2017'] = pd.merge(left, right, left_on ='ancienne_region', right_on='nom_ancienne_region', how='left').set_index('id')

# pour 2021
left = df['sdc_extanded'][['campagne', 'ancienne_region']].loc[
    df['sdc_extanded']['campagne'] == 2021
].reset_index()
right = df['agreste_ift_gcpe_reference_region_2021'][['nom_ancienne_region', 'contribution_ift_corrigee']]
df['sdc_extanded_2021'] = pd.merge(left, right, left_on ='ancienne_region', right_on='nom_ancienne_region', how='left').set_index('id')

In [331]:
#===================#
# pour les réalisés #
#===================#
df['sdc_realise'] = df['sdc_extanded'].loc[
    df['sdc_extanded']['entite_retenue'] == 'realise_retenu'
]

left = df['sdc_realise'].reset_index()
right = df['sdc_realise_performance'][['sdc_id']+[IFT_INDICATOR_AGROSYST]]
df['sdc_realise_extanded'] = pd.merge(left, right, left_on='id', right_on='sdc_id', how='left').set_index('id')

left = df['sdc_realise_extanded']
right = df['sdc_extanded_2017'][['contribution_ift_corrigee']].rename(columns={
    'contribution_ift_corrigee': 'ift_region_pk'
})
df['sdc_realise_extanded_2017'] = pd.merge(left, right, left_index=True, right_index=True, how='inner')

left = df['sdc_realise_extanded']
right = df['sdc_extanded_2021'][['contribution_ift_corrigee']].rename(columns={
    'contribution_ift_corrigee': 'ift_region_pk'
})
df['sdc_realise_extanded_2021'] = pd.merge(left, right, left_index =True, right_index=True, how='inner')


# on peut concaténer car on sait qu'on a aucun chevauchement (car un sdc est associé à une campagne, soit 2017, soit 2021)
df['sdc_realise_extanded'] = pd.concat([
    df['sdc_realise_extanded_2017'],
    df['sdc_realise_extanded_2021']
]).dropna()

#======================#
# pour les synthétisés #
#======================#
df['sdc_synthetise'] = df['sdc_extanded'].loc[
    (df['sdc_extanded']['entite_retenue'] != 'realise_retenu') & 
    ~(df['sdc_extanded']['entite_retenue'].isna())
] # on supprime les lignes pour lesquelles on a pas d'IFT (on a pas l'info pour toutes les régions !)



left = df['sdc_synthetise'].reset_index()
right = df['synthetise_synthetise_performance'][['synthetise_id']+[IFT_INDICATOR_AGROSYST]]
df['sdc_synthetise_extanded'] = pd.merge(left, right, left_on='entite_retenue', right_on='synthetise_id', how='left').set_index('id')

left = df['sdc_synthetise_extanded']
right = df['sdc_extanded_2017'][['contribution_ift_corrigee']].rename(columns={
    'contribution_ift_corrigee': 'ift_region_pk'
})
df['sdc_synthetise_extanded_2017'] = pd.merge(left, right, left_on ='sdc_id', right_index=True, how='inner')

left = df['sdc_synthetise_extanded']
right = df['sdc_extanded_2021'][['contribution_ift_corrigee']].rename(columns={
    'contribution_ift_corrigee': 'ift_region_pk'
})
df['sdc_synthetise_extanded_2021'] = pd.merge(left, right, left_on ='sdc_id', right_index=True, how='inner')

# on peut concaténer car on sait qu'on a aucun chevauchement (car un sdc est associé à une campagne, soit 2017, soit 2021)
df['sdc_synthetise_extanded'] = pd.concat([
    df['sdc_synthetise_extanded_2017'],
    df['sdc_synthetise_extanded_2021']
]).dropna()

#=========#
# commmun #
#=========#
df['sdc_extanded'] = pd.concat([
    df['sdc_realise_extanded'][['ift_cible_non_mil_chimique_tot', 'ift_region_pk']],
    df['sdc_synthetise_extanded'][['ift_cible_non_mil_chimique_tot', 'ift_region_pk']]
])

# ift norme : 
# si ift region > ift sdc --> ift_norme < 1
df['sdc_extanded']['ift_norme'] = df['sdc_extanded']['ift_cible_non_mil_chimique_tot'] / df['sdc_extanded']['ift_region_pk']

res = df['sdc_extanded'][['ift_region_pk', 'ift_norme']]

In [332]:
res

,ift_region_pk,ift_norme
id,,
fr.inra.agrosyst.api.entities.GrowingSystem_4205a378-7843-437b-8ba2-c8f95bb6cd96,5.20,3.772512
fr.inra.agrosyst.api.entities.GrowingSystem_b9734c41-1f93-4153-89cb-12e64d6a5782,2.10,0.390586
fr.inra.agrosyst.api.entities.GrowingSystem_14f26e2e-0db7-4a4b-97e5-777412abcafb,3.02,0.714168
fr.inra.agrosyst.api.entities.GrowingSystem_dacaf48d-5a9a-4529-ac03-70aac7a5effd,3.97,0.483903
fr.inra.agrosyst.api.entities.GrowingSystem_177d7239-0a10-4fb9-9d89-597863ca8035,2.68,1.389109
...,...,...
fr.inra.agrosyst.api.entities.GrowingSystem_d21ed657-4310-4880-bf85-cac59b3d5292,4.16,1.297814
fr.inra.agrosyst.api.entities.GrowingSystem_7772258a-7b50-40fe-b4c5-f77ead24cd50,4.09,2.110570
fr.inra.agrosyst.api.entities.GrowingSystem_07a329c8-2b0a-42d5-8ab1-f4b7080462c2,5.51,0.586735


,ift_region_pk,ift_norme
id,,
fr.inra.agrosyst.api.entities.GrowingSystem_4205a378-7843-437b-8ba2-c8f95bb6cd96,5.20,3.772512
fr.inra.agrosyst.api.entities.GrowingSystem_b9734c41-1f93-4153-89cb-12e64d6a5782,2.10,0.390586
fr.inra.agrosyst.api.entities.GrowingSystem_14f26e2e-0db7-4a4b-97e5-777412abcafb,3.02,0.714168
fr.inra.agrosyst.api.entities.GrowingSystem_dacaf48d-5a9a-4529-ac03-70aac7a5effd,3.97,0.483903
fr.inra.agrosyst.api.entities.GrowingSystem_177d7239-0a10-4fb9-9d89-597863ca8035,2.68,1.389109
...,...,...
fr.inra.agrosyst.api.entities.GrowingSystem_d21ed657-4310-4880-bf85-cac59b3d5292,4.16,1.297814
fr.inra.agrosyst.api.entities.GrowingSystem_7772258a-7b50-40fe-b4c5-f77ead24cd50,4.09,2.110570
fr.inra.agrosyst.api.entities.GrowingSystem_07a329c8-2b0a-42d5-8ab1-f4b7080462c2,5.51,0.586735


In [325]:
df['sdc_extanded']['ift_norme'].isna().value_counts()

ift_norme
False    5235
True        6
Name: count, dtype: int64

In [262]:
df['sdc_synthetise_extanded_2021']

,campagne,filiere,dispositif_id,domaine_id,commune_id,ancienne_region,sdc_id,entite_retenue,synthetise_id,ift_cible_non_mil_chimique_tot,ift_region_pk
id,,,,,,,,,,,
fr.inra.agrosyst.api.entities.GrowingSystem_2daff579-9169-4900-a665-08acabf1f85d,2021,GRANDES_CULTURES,fr.inra.agrosyst.api.entities.GrowingPlan_0013...,fr.inra.agrosyst.api.entities.Domain_7e5bcd39-...,fr.inra.agrosyst.api.entities.referential.RefL...,Centre-Val de Loire,fr.inra.agrosyst.api.entities.GrowingSystem_2d...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,5.400000,5.47
fr.inra.agrosyst.api.entities.GrowingSystem_d0e60b5c-981a-4c7c-a2d7-c9f6283d7dcd,2021,HORTICULTURE,fr.inra.agrosyst.api.entities.GrowingPlan_0031...,fr.inra.agrosyst.api.entities.Domain_836f285d-...,fr.inra.agrosyst.api.entities.referential.RefL...,Pays de la Loire,fr.inra.agrosyst.api.entities.GrowingSystem_d0...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,0.280857,4.09
fr.inra.agrosyst.api.entities.GrowingSystem_d6f31c33-49f2-43bb-b834-ebf93db50067,2021,GRANDES_CULTURES,fr.inra.agrosyst.api.entities.GrowingPlan_0036...,fr.inra.agrosyst.api.entities.Domain_301350af-...,fr.inra.agrosyst.api.entities.referential.RefL...,Midi-Pyrénées,fr.inra.agrosyst.api.entities.GrowingSystem_d6...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,1.251613,2.97
fr.inra.agrosyst.api.entities.GrowingSystem_e5541ab7-bbde-4aea-857e-6dcf267bddae,2021,GRANDES_CULTURES,fr.inra.agrosyst.api.entities.GrowingPlan_003f...,fr.inra.agrosyst.api.entities.Domain_473dd673-...,fr.inra.agrosyst.api.entities.referential.RefL...,Lorraine,fr.inra.agrosyst.api.entities.GrowingSystem_e5...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,4.157649,4.22
fr.inra.agrosyst.api.entities.GrowingSystem_6aa89004-df96-40a9-8f8b-6819a9757ce6,2021,POLYCULTURE_ELEVAGE,fr.inra.agrosyst.api.entities.GrowingPlan_0058...,fr.inra.agrosyst.api.entities.Domain_3761590f-...,fr.inra.agrosyst.api.entities.referential.RefL...,Pays de la Loire,fr.inra.agrosyst.api.entities.GrowingSystem_6a...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,0.000000,4.09
...,...,...,...,...,...,...,...,...,...,...,...
fr.inra.agrosyst.api.entities.GrowingSystem_d21ed657-4310-4880-bf85-cac59b3d5292,2021,GRANDES_CULTURES,fr.inra.agrosyst.api.entities.GrowingPlan_ff9d...,fr.inra.agrosyst.api.entities.Domain_c8eea478-...,fr.inra.agrosyst.api.entities.referential.RefL...,Poitou-Charentes,fr.inra.agrosyst.api.entities.GrowingSystem_d2...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,5.398906,4.16
fr.inra.agrosyst.api.entities.GrowingSystem_7772258a-7b50-40fe-b4c5-f77ead24cd50,2021,VITICULTURE,fr.inra.agrosyst.api.entities.GrowingPlan_ffb8...,fr.inra.agrosyst.api.entities.Domain_1b237994-...,fr.inra.agrosyst.api.entities.referential.RefL...,Pays de la Loire,fr.inra.agrosyst.api.entities.GrowingSystem_77...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,8.632233,4.09
fr.inra.agrosyst.api.entities.GrowingSystem_07a329c8-2b0a-42d5-8ab1-f4b7080462c2,2021,POLYCULTURE_ELEVAGE,fr.inra.agrosyst.api.entities.GrowingPlan_ffba...,fr.inra.agrosyst.api.entities.Domain_15c556ee-...,fr.inra.agrosyst.api.entities.referential.RefL...,Basse-Normandie,fr.inra.agrosyst.api.entities.GrowingSystem_07...,fr.inra.agrosyst.api.entities.practiced.Practi...,fr.inra.agrosyst.api.entities.practiced.Practi...,3.232908,5.51


TypeError: 'set' object is not callable

In [217]:
df['sdc_realise_extanded']

,sdc_id,campagne_x,filiere,dispositif_id,domaine_id,commune_id,ancienne_region_x,sdc_id_x,entite_retenue,sdc_id_y,ift_cible_non_mil_chimique_tot,campagne_y,ancienne_region_y,nom_ancienne_region,contribution_ift_corrigee_x,contribution_ift_corrigee_y,contribution_ift_corrigee
1020.0,fr.inra.agrosyst.api.entities.GrowingSystem_42...,2017,MARAICHAGE,fr.inra.agrosyst.api.entities.GrowingPlan_000d...,fr.inra.agrosyst.api.entities.Domain_3e4af173-...,fr.inra.agrosyst.api.entities.referential.RefL...,Centre-Val de Loire,fr.inra.agrosyst.api.entities.GrowingSystem_42...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_42...,19.617061,2017.0,Centre-Val de Loire,Centre-Val de Loire,5.2,5.2,5.2
8804.0,fr.inra.agrosyst.api.entities.GrowingSystem_05...,2021,POLYCULTURE_ELEVAGE,fr.inra.agrosyst.api.entities.GrowingPlan_0014...,fr.inra.agrosyst.api.entities.Domain_6fe285bc-...,fr.inra.agrosyst.api.entities.referential.RefL...,Poitou-Charentes,fr.inra.agrosyst.api.entities.GrowingSystem_05...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_05...,2.470542,NaN,NaN,NaN,NaN,NaN,NaN
8470.0,fr.inra.agrosyst.api.entities.GrowingSystem_a5...,2022,POLYCULTURE_ELEVAGE,fr.inra.agrosyst.api.entities.GrowingPlan_0015...,fr.inra.agrosyst.api.entities.Domain_c3767a53-...,fr.inra.agrosyst.api.entities.referential.RefL...,Franche-Comté,fr.inra.agrosyst.api.entities.GrowingSystem_a5...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_a5...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
611.0,fr.inra.agrosyst.api.entities.GrowingSystem_bb...,2018,HORTICULTURE,fr.inra.agrosyst.api.entities.GrowingPlan_0018...,fr.inra.agrosyst.api.entities.Domain_d74e1a2e-...,fr.inra.agrosyst.api.entities.referential.RefL...,Aquitaine,fr.inra.agrosyst.api.entities.GrowingSystem_bb...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_bb...,0.812030,NaN,NaN,NaN,NaN,NaN,NaN
8698.0,fr.inra.agrosyst.api.entities.GrowingSystem_d2...,2022,GRANDES_CULTURES,fr.inra.agrosyst.api.entities.GrowingPlan_0018...,fr.inra.agrosyst.api.entities.Domain_b05aecd9-...,fr.inra.agrosyst.api.entities.referential.RefL...,Centre-Val de Loire,fr.inra.agrosyst.api.entities.GrowingSystem_d2...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_d2...,3.334462,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6813.0,fr.inra.agrosyst.api.entities.GrowingSystem_97...,2018,GRANDES_CULTURES,fr.inra.agrosyst.api.entities.GrowingPlan_ffde...,fr.inra.agrosyst.api.entities.Domain_71238793-...,fr.inra.agrosyst.api.entities.referential.RefL...,Champagne-Ardenne,fr.inra.agrosyst.api.entities.GrowingSystem_97...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_97...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
3046.0,fr.inra.agrosyst.api.entities.GrowingSystem_26...,2014,POLYCULTURE_ELEVAGE,fr.inra.agrosyst.api.entities.GrowingPlan_ffe1...,fr.inra.agrosyst.api.entities.Domain_e04f72db-...,fr.inra.agrosyst.api.entities.referential.RefL...,Nord-Pas-de-Calais,fr.inra.agrosyst.api.entities.GrowingSystem_26...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_26...,3.888511,NaN,NaN,NaN,NaN,NaN,NaN
NaN,fr.inra.agrosyst.api.entities.GrowingSystem_09...,2023,GRANDES_CULTURES,fr.inra.agrosyst.api.entities.GrowingPlan_ffe8...,fr.inra.agrosyst.api.entities.Domain_4413d2db-...,fr.inra.agrosyst.api.entities.referential.RefL...,Poitou-Charentes,fr.inra.agrosyst.api.entities.GrowingSystem_09...,realise_retenu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6480.0,fr.inra.agrosyst.api.entities.GrowingSystem_23...,2020,POLYCULTURE_ELEVAGE,fr.inra.agrosyst.api.entities.GrowingPlan_fff4...,fr.inra.agrosyst.api.entities.Domain_329c03cd-...,fr.inra.agrosyst.api.entities.referential.RefL...,Bretagne,fr.inra.agrosyst.api.entities.GrowingSystem_23...,realise_retenu,fr.inra.agrosyst.api.entities.GrowingSystem_23...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [162]:
df['sdc_extanded']

,ancienne_region,nom_ancienne_region,contribution_ift_corrigee
id,,,
fr.inra.agrosyst.api.entities.GrowingSystem_82957eec-02aa-4eea-88b8-4fda93bdec28,Midi-Pyrénées,Midi-Pyrénées,3.02
fr.inra.agrosyst.api.entities.GrowingSystem_df53bb47-1925-4589-b4a5-9845626535b0,Pays de la Loire,Pays de la Loire,3.89
fr.inra.agrosyst.api.entities.GrowingSystem_671725aa-074f-4385-bab5-483fac0f1d5f,Franche-Comté,Franche-Comté,4.53
fr.inra.agrosyst.api.entities.GrowingSystem_4205a378-7843-437b-8ba2-c8f95bb6cd96,Centre-Val de Loire,Centre-Val de Loire,5.20
fr.inra.agrosyst.api.entities.GrowingSystem_61a359db-9b28-4695-8f97-0b281ab30d5d,Pays de la Loire,Pays de la Loire,3.89
...,...,...,...
fr.inra.agrosyst.api.entities.GrowingSystem_0c584efa-7693-452e-ab1f-fa85e32825c5,Poitou-Charentes,Poitou-Charentes,4.33
fr.inra.agrosyst.api.entities.GrowingSystem_739057ad-d239-4fca-92d1-f5b9e6eef4bc,Corse,NaN,NaN
fr.inra.agrosyst.api.entities.GrowingSystem_6df8688e-4130-4974-af70-61511db8d60b,Haute-Normandie,Haute-Normandie,6.11


In [ ]:
# Provence-Alpes-Côte d'Azur
# Centre-Val de Loire  
# Île-de-France

In [ ]:
df['sdc_extanded']

,ancienne_region,nom_ancienne_region,contribution_ift_corrigee
0,Midi-Pyrénées,Midi-Pyrénées,3.02
1,Pays de la Loire,Pays de la Loire,3.89
2,Franche-Comté,Franche-Comté,4.53
3,Centre-Val de Loire,Centre-Val de Loire,5.20
4,Pays de la Loire,Pays de la Loire,3.89
...,...,...,...
39829,Poitou-Charentes,Poitou-Charentes,4.33
39830,Corse,NaN,NaN
39831,Haute-Normandie,Haute-Normandie,6.11
39832,Poitou-Charentes,Poitou-Charentes,4.33
